In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import json
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

import warnings
warnings.filterwarnings('ignore')


In [7]:
# (CHANGE THESE ACCORDING TO YOUR DRIVE)
STUDENTLIFE_PATH = "/content/drive/MyDrive/burnoutcode1"
#OUTPUT_PATH = "/content/drive/MyDrive/AI_Burnout_Predictor/results_realistic_studentlife"

#os.makedirs(OUTPUT_PATH, exist_ok=True)

print("Paths configured.")


Paths configured.


In [8]:
# Loading StudentLife data
def load_studentlife_json(folder_path):
    all_data = []
    for file_name in os.listdir(folder_path):
        if file_name.endswith(".json"):
            student_id = file_name.replace(".json", "")
            with open(os.path.join(folder_path, file_name), "r") as f:
                records = json.load(f)
                for r in records:
                    r["student_id"] = student_id
                    all_data.append(r)
    return pd.DataFrame(all_data)

print("Loading StudentLife...")

stress_raw = load_studentlife_json(os.path.join(STUDENTLIFE_PATH, "Stress"))
activity_raw = load_studentlife_json(os.path.join(STUDENTLIFE_PATH, "Activity"))
sleep_raw = load_studentlife_json(os.path.join(STUDENTLIFE_PATH, "Sleep"))

print(f"Stress: {stress_raw.shape}")
print(f"Activity: {activity_raw.shape}")
print(f"Sleep: {sleep_raw.shape}")


Loading StudentLife...
Stress: (2408, 5)
Activity: (833, 9)
Sleep: (1644, 7)


In [9]:
stress_raw.head()

,null,resp_time,student_id,level,location
0,"43.70477575,-72.28844073",1364121982,Stress_u44,NaN,NaN
1,2,1364121983,Stress_u44,NaN,NaN
2,"43.70637091,-72.28704334",1364118696,Stress_u44,NaN,NaN
3,4,1364121980,Stress_u44,NaN,NaN
4,3,1364121985,Stress_u44,NaN,NaN


In [10]:
activity_raw.head()

,Social2,null,resp_time,student_id,other_relaxing,other_working,relaxing,working,location
0,1,1,1364678236,Activity_u52,NaN,NaN,NaN,NaN,NaN
1,2,1,1364430105,Activity_u52,NaN,NaN,NaN,NaN,NaN
2,2,1,1364504412,Activity_u52,NaN,NaN,NaN,NaN,NaN
3,2,1,1364592864,Activity_u52,NaN,NaN,NaN,NaN,NaN
4,2,1,1364592869,Activity_u52,NaN,NaN,NaN,NaN,NaN


In [11]:
sleep_raw.head()

,null,resp_time,student_id,hour,location,rate,social
0,"43.70641977,-72.28711458",1364118747,Sleep_u52,NaN,NaN,NaN,NaN
1,1,1364121887,Sleep_u52,NaN,NaN,NaN,NaN
2,8,1364121890,Sleep_u52,NaN,NaN,NaN,NaN
3,"43.70427927,-72.28971705",1364121889,Sleep_u52,NaN,NaN,NaN,NaN
4,1,1364121891,Sleep_u52,NaN,NaN,NaN,NaN


In [12]:
# Cleaning the STRESS DATASET
# =========================
print("\n DATASET: STRESS")
print("Shows self-reported student stress levels over time")

print("\nCleaning StudentLife Stress...")

stress_clean = stress_raw.copy()

# Dropping the 'null' column
if 'null' in stress_clean.columns:
    stress_clean = stress_clean.drop(columns=['null'])

print("\nInitial Stress Dataset:")
print(stress_clean)

# Converting  timestamp
stress_clean['timestamp'] = pd.to_datetime(stress_clean['resp_time'], unit='s')
print("\nAfter converting resp_time to timestamp:")
print(stress_clean)

# Cleaning  student_id
stress_clean['student_id'] = stress_clean['student_id'].str.replace('Stress_', '')
print("\nAfter cleaning student_id:")
print(stress_clean)



 DATASET: STRESS
Shows self-reported student stress levels over time

Cleaning StudentLife Stress...

Initial Stress Dataset:
       resp_time  student_id level                  location
0     1364121982  Stress_u44   NaN                       NaN
1     1364121983  Stress_u44   NaN                       NaN
2     1364118696  Stress_u44   NaN                       NaN
3     1364121980  Stress_u44   NaN                       NaN
4     1364121985  Stress_u44   NaN                       NaN
...          ...         ...   ...                       ...
2403  1369200175  Stress_u36     4   43.7069299,-72.28719159
2404  1369164593  Stress_u36     4   43.70681113,-72.2870305
2405  1369311300  Stress_u36     1  43.70662411,-72.28722922
2406  1369351445  Stress_u36     1  43.65172993,-72.31003051
2407  1369373276  Stress_u36     1   43.70331517,-72.2903492

[2408 rows x 4 columns]

After converting resp_time to timestamp:
       resp_time  student_id level                  location  \
0     1364

In [13]:
#renaming the level column
stress_clean = stress_clean.rename(columns={'level': 'stress_level'})
print("\nAfter renaming level → stress_level:")
print(stress_clean)

#Converting stress_level to numeric data
stress_clean['stress_level'] = pd.to_numeric(stress_clean['stress_level'], errors='coerce')
print("\nAfter converting stress_level to numeric:")
print(stress_clean)

if 'null' in stress_clean.columns:

    # if stress_level is missing but null looks like numeric, then use this
    null_as_num = pd.to_numeric(stress_clean['null'], errors='coerce')
    fill_mask = stress_clean['stress_level'].isna() & null_as_num.notna()
    if fill_mask.any():
        stress_clean.loc[fill_mask, 'stress_level'] = null_as_num.loc[fill_mask]

    #If location is missing but null looks like "lat,long", then use this
    if 'location' in stress_clean.columns:
        null_as_str = stress_clean['null'].astype(str)
        coord_mask = stress_clean['location'].isna() & null_as_str.str.match(
            r'^-?\d+(\.\d+)?,-?\d+(\.\d+)?$'
        )
        if coord_mask.any():
            stress_clean.loc[coord_mask, 'location'] = stress_clean.loc[coord_mask, 'null']

print("\nAfter recovering values from 'null' (if applicable):")
print(stress_clean)

#dropping missing stress values
stress_clean = stress_clean.dropna(subset=['stress_level'])
print("\nAfter dropping NaN stress levels:")
print(stress_clean)

#Explicit float conversion
stress_clean['stress_level'] = stress_clean['stress_level'].astype(float)
print("\nFinal cleaned Stress dataset:")
print(stress_clean)



After renaming level → stress_level:
       resp_time student_id stress_level                  location  \
0     1364121982        u44          NaN                       NaN   
1     1364121983        u44          NaN                       NaN   
2     1364118696        u44          NaN                       NaN   
3     1364121980        u44          NaN                       NaN   
4     1364121985        u44          NaN                       NaN   
...          ...        ...          ...                       ...   
2403  1369200175        u36            4   43.7069299,-72.28719159   
2404  1369164593        u36            4   43.70681113,-72.2870305   
2405  1369311300        u36            1  43.70662411,-72.28722922   
2406  1369351445        u36            1  43.65172993,-72.31003051   
2407  1369373276        u36            1   43.70331517,-72.2903492   

               timestamp  
0    2013-03-24 10:46:22  
1    2013-03-24 10:46:23  
2    2013-03-24 09:51:36  
3    2013-03-

In [14]:
# Code for Activity data cleaning
print("\n DATASET: ACTIVITY")
print(" Shows students' daily activity levels (social, working, relaxing)")

print("\nCleaning StudentLife Activity")

activity_clean = activity_raw.copy()

# Drop the 'null' coloumns
if 'null' in activity_clean.columns:
    activity_clean = activity_clean.drop(columns=['null'])

print("\nInitial Activity Dataset:")
print(activity_clean)

# Convert timestamp

activity_clean['timestamp'] = pd.to_datetime(activity_clean['resp_time'], unit='s')
print("\nAfter converting resp_time to timestamp:")
print(activity_clean)

# Clean student_id
activity_clean['student_id'] = activity_clean['student_id'].str.replace('Activity_', '')
print("\nAfter cleaning student_id:")
print(activity_clean)



 DATASET: ACTIVITY
 Shows students' daily activity levels (social, working, relaxing)

Cleaning StudentLife Activity

Initial Activity Dataset:
    Social2   resp_time    student_id other_relaxing other_working relaxing  \
0         1  1364678236  Activity_u52            NaN           NaN      NaN   
1         2  1364430105  Activity_u52            NaN           NaN      NaN   
2         2  1364504412  Activity_u52            NaN           NaN      NaN   
3         2  1364592864  Activity_u52            NaN           NaN      NaN   
4         2  1364592869  Activity_u52            NaN           NaN      NaN   
..      ...         ...           ...            ...           ...      ...   
828     NaN  1366671689  Activity_u04              2             2        3   
829     NaN  1367175614  Activity_u04              3             3        3   
830     NaN  1367083593  Activity_u04              3             2        2   
831     NaN  1366997352  Activity_u04              2             

In [17]:


# key columns of studentlife dataset

for col in ["Social2", "working", "other_working", "relaxing", "other_relaxing"]:
    if col not in activity_clean.columns:
        activity_clean[col] = np.nan



# Converting to numeric

for col in ["Social2", "working", "other_working", "relaxing", "other_relaxing"]:
  activity_clean[col] = pd.to_numeric(activity_clean[col], errors="coerce")



# Computing interpretable scores

activity_clean["workload_score"] = activity_clean[["working", "other_working"]].sum(axis=1, min_count=1)
activity_clean["recovery_score"] = activity_clean[["relaxing", "other_relaxing"]].sum(axis=1, min_count=1)
activity_clean["social_score"] = activity_clean["Social2"]


print("\nAfter computing workload_score / recovery_score / social_score:")

print(activity_clean[["student_id", "timestamp", "workload_score", "recovery_score", "social_score"]].head())



# Keep relevant columns and drop rows where ALL scores are missing

activity_clean = activity_clean[["student_id", "timestamp", "workload_score", "recovery_score", "social_score"]]
activity_clean = activity_clean.dropna( subset=["workload_score", "recovery_score", "social_score"], how="all" ).copy()



print("\nFinal cleaned Activity dataset (3 scores):")
print(activity_clean.head())




After computing workload_score / recovery_score / social_score:
  student_id           timestamp  workload_score  recovery_score  social_score
0        u52 2013-03-30 21:17:16             NaN             NaN           1.0
1        u52 2013-03-28 00:21:45             NaN             NaN           2.0
2        u52 2013-03-28 21:00:12             NaN             NaN           2.0
3        u52 2013-03-29 21:34:24             NaN             NaN           2.0
4        u52 2013-03-29 21:34:29             NaN             NaN           2.0

Final cleaned Activity dataset (3 scores):
  student_id           timestamp  workload_score  recovery_score  social_score
0        u52 2013-03-30 21:17:16             NaN             NaN           1.0
1        u52 2013-03-28 00:21:45             NaN             NaN           2.0
2        u52 2013-03-28 21:00:12             NaN             NaN           2.0
3        u52 2013-03-29 21:34:24             NaN             NaN           2.0
4        u52 2013-03-2

In [15]:
# Sleep dataset processing

# Print dataset name
print("\ DATASET: SLEEP")

# Describe dataset purpose
print("Shows students' self-reported sleep duration in hours")

# Start cleaning process
print("\nCleaning StudentLife Sleep...")

# Create a copy of raw dataset
sleep_clean = sleep_raw.copy()

# Remove irrelevant null column if present
if 'null' in sleep_clean.columns:
    sleep_clean = sleep_clean.drop(columns=['null'])

# Display initial dataset
print("\nInitial Sleep Dataset:")
print(sleep_clean)

# Convert response time to timestamp
sleep_clean['timestamp'] = pd.to_datetime(sleep_clean['resp_time'], unit='s')

# Show dataset after timestamp conversion
print("\nAfter converting resp_time to timestamp:")
print(sleep_clean)

\ DATASET: SLEEP
Shows students' self-reported sleep duration in hours

Cleaning StudentLife Sleep...

Initial Sleep Dataset:
       resp_time student_id hour                  location rate social
0     1364118747  Sleep_u52  NaN                       NaN  NaN    NaN
1     1364121887  Sleep_u52  NaN                       NaN  NaN    NaN
2     1364121890  Sleep_u52  NaN                       NaN  NaN    NaN
3     1364121889  Sleep_u52  NaN                       NaN  NaN    NaN
4     1364121891  Sleep_u52  NaN                       NaN  NaN    NaN
...          ...        ...  ...                       ...  ...    ...
1639  1366167770  Sleep_u15    8  43.70591041,-72.28826857    1      1
1640  1366049005  Sleep_u15    6  43.70808053,-72.28506786    1      1
1641  1366167725  Sleep_u15    9  43.70591041,-72.28826857    1      1
1642  1366653844  Sleep_u15    8  43.70746352,-72.28565777    1      1
1643  1369847081  Sleep_u15    4                   Unknown    4      1

[1644 rows x 6 column

In [16]:
# Clean student_id by removing prefix
sleep_clean['student_id'] = sleep_clean['student_id'].str.replace('Sleep_', '')

# Show dataset after cleaning student_id
print("\nAfter cleaning student_id:")
print(sleep_clean)

# Convert hour column to numeric sleep_hours
sleep_clean['sleep_hours'] = pd.to_numeric(sleep_clean['hour'], errors='coerce')

# Show dataset after converting sleep hours
print("\nAfter converting hour to sleep_hours:")
print(sleep_clean)

# Recover missing sleep hours from null column if available
if 'null' in sleep_clean.columns:
    null_as_num = pd.to_numeric(sleep_clean['null'], errors='coerce')
    fill_mask = sleep_clean['sleep_hours'].isna() & null_as_num.notna()
    if fill_mask.any():
        sleep_clean.loc[fill_mask, 'sleep_hours'] = null_as_num.loc[fill_mask]

# Show dataset after recovery step
print("\nAfter recovering sleep_hours from null if applicable:")
print(sleep_clean)

# Keep only relevant columns and remove missing values
sleep_clean = sleep_clean[['student_id', 'timestamp', 'sleep_hours']].dropna()

# Display final cleaned dataset
print("\nFinal cleaned Sleep dataset:")
print(sleep_clean)


After cleaning student_id:
       resp_time student_id hour                  location rate social  \
0     1364118747        u52  NaN                       NaN  NaN    NaN   
1     1364121887        u52  NaN                       NaN  NaN    NaN   
2     1364121890        u52  NaN                       NaN  NaN    NaN   
3     1364121889        u52  NaN                       NaN  NaN    NaN   
4     1364121891        u52  NaN                       NaN  NaN    NaN   
...          ...        ...  ...                       ...  ...    ...   
1639  1366167770        u15    8  43.70591041,-72.28826857    1      1   
1640  1366049005        u15    6  43.70808053,-72.28506786    1      1   
1641  1366167725        u15    9  43.70591041,-72.28826857    1      1   
1642  1366653844        u15    8  43.70746352,-72.28565777    1      1   
1643  1369847081        u15    4                   Unknown    4      1   

               timestamp  
0    2013-03-24 09:52:27  
1    2013-03-24 10:44:47  
2 